# GlowLab · 03 — Baholash, confusion matrix, threshold, Grad-CAM

1. **Test accuracy** va har sinf bo'yicha aniqlik
2. **Confusion matrix** — model qaysi sinflarni adashtiryapti
3. **Threshold** — model ishonchi past bo'lsa ilova "Ishonchim komil emas" deydi. Chegarani **val** to'plamidan tanlaymiz
4. **Grad-CAM** — model rasmning qaysi qismiga qarab qaror qildi (AI Lab sahifasi uchun rasmlar)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/glowlab_ml'
DATA, OUT = f'{ROOT}/data', f'{ROOT}/models'

import json, os
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
CLASSES = json.load(open(f'{OUT}/classes.json'))        # O'qitishdagi tartib
IMG_SIZE, MEAN, STD = 224, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
eval_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

model = models.mobilenet_v2(weights=None)                # Bo'sh model
model.classifier[1] = nn.Linear(model.last_channel, len(CLASSES))
model.load_state_dict(torch.load(f'{OUT}/product_classifier.pt', map_location=device))  # Saqlangan og'irliklar
model = model.to(device).eval()

In [ ]:
def predict_all(split):
    """Butun to'plam bo'yicha: haqiqiy sinf, ehtimollar"""
    ds = datasets.ImageFolder(f'{DATA}/{split}', transform=eval_tf)
    remap = {ds.class_to_idx[c]: i for i, c in enumerate(CLASSES)}
    dl = torch.utils.data.DataLoader(ds, batch_size=64)
    ys, probs = [], []
    with torch.no_grad():
        for x, y in dl:
            p = F.softmax(model(x.to(device)), dim=1).cpu()     # softmax → ehtimollar (yig'indisi 1)
            probs.append(p)
            ys.append(torch.tensor([remap[int(t)] for t in y]))
    return torch.cat(ys).numpy(), torch.cat(probs).numpy()

y_test, p_test = predict_all('test')
pred = p_test.argmax(1)
print('TEST accuracy:', round((pred == y_test).mean(), 3))
for i, c in enumerate(CLASSES):
    mask = y_test == i
    print(f'  {c:12s} {(pred[mask] == i).mean():.3f}  (n={mask.sum()})')

In [ ]:
# Confusion matrix: qator = haqiqiy, ustun = bashorat
cm = np.zeros((len(CLASSES), len(CLASSES)), dtype=int)
for t, p in zip(y_test, pred):
    cm[t, p] += 1

plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap='Blues')
plt.xticks(range(len(CLASSES)), CLASSES, rotation=45, ha='right')
plt.yticks(range(len(CLASSES)), CLASSES)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        plt.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.xlabel('Bashorat'); plt.ylabel('Haqiqiy'); plt.title('Confusion matrix (test)')
plt.tight_layout()
plt.savefig(f'{OUT}/confusion_matrix.png', dpi=150)       # AI Lab sahifasi uchun
plt.show()

## Threshold tanlash (val to'plamida!)
Maqsad: model **javob bergan** holatlarda kamida **90%** to'g'ri bo'lsin. Ishonchi chegaradan past bo'lsa — "Ishonchim komil emas".
Test to'plamida tanlamaymiz — aks holda test natijasi "aldangan" bo'ladi.

In [ ]:
y_val, p_val = predict_all('val')
conf_val, pred_val = p_val.max(1), p_val.argmax(1)

TARGET = 0.90
threshold = 0.5
for t in np.arange(0.30, 0.96, 0.01):
    keep = conf_val >= t                                   # Shu chegaradan yuqorilari "javob beriladi"
    if keep.sum() < 10:
        break
    acc = (pred_val[keep] == y_val[keep]).mean()
    if acc >= TARGET:
        threshold = float(round(t, 2))
        break

keep_test = p_test.max(1) >= threshold
print('threshold:', threshold)
print('test: javob berildi', round(keep_test.mean(), 3), '| ulardagi aniqlik', round((pred[keep_test] == y_test[keep_test]).mean(), 3))

## Grad-CAM
Oxirgi konvolyutsion qatlam (`features[-1]`) faolligi va gradientidan issiqlik xaritasi.

In [ ]:
from PIL import Image

acts, grads = {}, {}
layer = model.features[-1]
def save_act(module, inp, out):
    acts['a'] = out                                          # Faollik (activation)
    out.register_hook(lambda g: grads.__setitem__('g', g))   # Shu tensorning gradienti (backward paytida)
layer.register_forward_hook(save_act)

def grad_cam(path):
    img = Image.open(path).convert('RGB')
    x = eval_tf(img).unsqueeze(0).to(device)               # [1, 3, 224, 224]
    model.zero_grad()
    out = model(x)
    cls = out.argmax(1).item()
    out[0, cls].backward()                                  # Tanlangan sinf bo'yicha gradient
    w = grads['g'].mean(dim=(2, 3), keepdim=True)           # Har kanal "muhimligi"
    cam = F.relu((w * acts['a']).sum(1)).squeeze()          # Og'irlikli yig'indi → faqat musbat
    cam = (cam / (cam.max() + 1e-8)).detach().cpu().numpy()
    cam = np.array(Image.fromarray(np.uint8(cam * 255)).resize(img.size)) / 255.0
    return img, cam, CLASSES[cls], F.softmax(out, 1)[0, cls].item()

os.makedirs(f'{OUT}/gradcam', exist_ok=True)
for c in CLASSES:
    name = sorted(os.listdir(f'{DATA}/test/{c}'))[0]          # Har sinfdan bitta misol
    img, cam, label, conf = grad_cam(f'{DATA}/test/{c}/{name}')
    plt.figure(figsize=(4, 4))
    plt.imshow(img); plt.imshow(cam, cmap='jet', alpha=0.4); plt.axis('off')
    plt.title(f'true {c} → {label} {conf:.0%}', fontsize=9)
    plt.savefig(f'{OUT}/gradcam/{c}.png', dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
metrics = {
    'test_accuracy': float((pred == y_test).mean()),
    'per_class': {c: float((pred[y_test == i] == i).mean()) for i, c in enumerate(CLASSES)},
    'threshold': threshold,
    'test_answered_rate': float(keep_test.mean()),
    'test_answered_accuracy': float((pred[keep_test] == y_test[keep_test]).mean()),
    'confusion_matrix': cm.tolist(),
    'classes': CLASSES,
}
json.dump(metrics, open(f'{OUT}/metrics.json', 'w'), indent=2)   # Ilova AI Lab sahifasida ko'rsatadi
print(json.dumps(metrics, indent=2))